# ErgoPose Risk Classifier — Model Training

This notebook is the **third stage** of the *ErgoPose Risk Classifier* project.  
It defines, trains, and evaluates the **Artificial Neural Network (ANN)** for posture classification based on the preprocessed dataset.

### Objectives
- Load the cleaned dataset from `data/processed/`.
- Encode categorical posture labels.
- Split the data into training and testing sets.
- Define and train an ANN model for multi-class classification.
- Save the trained model and scaler to the `models/` directory.

### Input and Output
- **Input:** `data/processed/clean_postural_risk_dataset.csv`  
- **Outputs:**  
  - `models/neural_network.pkl`  
  - `models/scaler.pkl`

In [1]:
"""
Imports the necessary libraries for model definition, training, and evaluation.
"""

# [1] Imports
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score
import joblib
from math import sqrt

import torch
from torch import Tensor, nn, optim

In [2]:
"""
Defines paths for processed data and model output directories.
"""

# [2] Paths configuration
DATA_PATH = Path("../data/processed/clean_postural_risk_dataset.csv")
MODELS_PATH = Path("../models")
MODELS_PATH.mkdir(exist_ok=True)

print(f"Dataset path: {DATA_PATH}")
print(f"Models directory: {MODELS_PATH}")


Dataset path: ../data/processed/clean_postural_risk_dataset.csv
Models directory: ../models


In [3]:
"""
Defines the number of neurons in the hidden layers by the 'Geometric Pyramid Rule'
"""

data = pd.read_csv(DATA_PATH)

X = data.drop(columns=['upperbody_label'])
y = data['upperbody_label']

input_neurons = X.shape[1]
output_neurons = 2 # Binary classification

input_output_neurons = int(sqrt(input_neurons*output_neurons))

print(f"The number of neurons in hidden layers will be: {int(input_output_neurons*0.5)} <= N <= {int(input_output_neurons*2)}")

The number of neurons in hidden layers will be: 5 <= N <= 20


In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Training data shape: (3355, 51)
Testing data shape: (1439, 51)


## Model Architecture Proposals - rules
- Hidden layers must have beetwen **5 and 20 neurons total**. If more than 1 hidden layer is implemented, the number of neurons of both layers must **add up** to a number beetwen **5 and 20**.
- Batch size, at this initial stage, must be **default**.
- Activation function **cannot** be tanh.
- Learning rate must be $10^{-2}, 10^{-3}$ or **smaller numbers**.

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))

Usando dispositivo: cuda
2.1.2+cu118
11.8
NVIDIA GeForce GTX 1050 Ti


In [9]:
class MLPModel(nn.Module):
    def __init__(self, input_dim, hidden_dim_1, hidden_dim_2, output_dim):
        super(MLPModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim_1),
            nn.ReLU(),
            nn.Linear(hidden_dim_1, hidden_dim_2),
            nn.ReLU(),
            nn.Linear(hidden_dim_2, output_dim),
        )

    def forward(self, x):
        return self.net(x)



X_train_t: Tensor = torch.tensor(X_train.values, dtype=torch.float32).to(device)
X_test_t: Tensor = torch.tensor(X_test.values, dtype=torch.float32).to(device)

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

y_train_t = torch.tensor(y_train_enc, dtype=torch.long).to(device)
y_test_t_cpu = y_test_enc

input_dim: int = X_train_t.shape[1]
hidden_dim_1 = 8
hidden_dim_2 = 10
output_dim = len(y.unique())

model: MLPModel = MLPModel(input_dim, hidden_dim_1, hidden_dim_2, output_dim).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(300):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()
    # print(f"Epoch {epoch}: loss {loss}")
model.eval()
with torch.no_grad():
    outputs = model(X_test_t)
    y_pred_t = torch.argmax(outputs, dim=1)
    y_pred = y_pred_t.cpu().numpy()

acc = accuracy_score(y_test_t_cpu, y_pred)
f1 = f1_score(y_test_t_cpu, y_pred)

print(f"Acurácia: {acc:.2f} - F1: {f1:.2f}")

Acurácia: 0.79 - F1: 0.65
